In [1]:
#import libraries
import pandas as pd
import requests
import re

In [2]:
def extract_make_model_year(title):
    pattern = r"(?P<make>\b[a-zA-Z\-]+)\s+(?P<model>[A-Za-z0-9\-]+(?:\s[A-Za-z0-9\-]+)?)?.*?(?P<year>\d{4})(?!\d)"
    match = re.search(pattern, title.strip())
    if match:
        return match.group('make'), match.group('model'), match.group('year')
    return None, None, None

In [3]:
def extract_condition(condition):
    condition_lower = condition.lower()
    if "foreign used" in condition_lower:
        return "foreign used"
    elif "local used" in condition_lower:
        return "local used"
    elif "brand new" in condition_lower:
        return "brand new"
    else:
        return None

In [14]:
def extract_transmission(transmission):
    transmission_lower = transmission.lower()
    if "automatic" in transmission_lower:
        return "automation"
    elif "manual" in transmission_lower:
        return "manual"
    else:
        return None

In [15]:
def fetch_data(page):
    url =  "https://jiji.ng/api_web/v1/listing"
    params = {
        "slug": "cars",
        "page": page,
        "webp": True
    }

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException:
        print(f"[page] {page}, Request error")
        return []
    except ValueError:
        print(f"[page] {page}. Decode error")

    adverts = data.get("adverts_list", {}).get("adverts", [])
    if not isinstance(adverts, list):
        print(f"An error occured. Expected list but got {type(adverts)}")
        return []

    return adverts

    

In [16]:
def get_attr_value(attrs, key_name):
    for attr in attrs:
        if attr.get("name", "").lower() == key_name.lower():
            return attr.get("value", "").strip()
    return None

In [17]:
def main():
    all_ads = []
    for page in range(1, 101):
        ads = fetch_data(page)
        print(f"Page {page}. {len(ads)} ads found")
        
        for ad in ads:
            if isinstance(ad, dict):
                attrs = ad.get("attrs", [])
                title = ad.get("title", "")
                condition_ = get_attr_value(attrs, "condition")
                transmission_ = get_attr_value(attrs, "transmission")
                condition = extract_condition(condition_)
                transmission = extract_transmission(transmission_)
                make, model, year = extract_make_model_year(title)
                price = ad.get("price_title", "")

                if price:
                    all_ads.append({
                        "title": title,
                        "make": make,
                        "model": model,
                        "year": year,
                        "condition": condition,
                        "transmission": transmission,
                        "price": price
                    })

    if all_ads:
        df = pd.DataFrame(all_ads)
        df.to_csv("jiji_car_dataset.csv", index=False)
        print("Successfully extract data from jiji")
    else:
        print("Nothing to extract.")



main()      

Page 1. 20 ads found
Page 2. 20 ads found
Page 3. 20 ads found
Page 4. 20 ads found
Page 5. 20 ads found
Page 6. 20 ads found
Page 7. 20 ads found
Page 8. 20 ads found
Page 9. 20 ads found
Page 10. 20 ads found
Page 11. 20 ads found
Page 12. 20 ads found
Page 13. 20 ads found
Page 14. 20 ads found
Page 15. 20 ads found
Page 16. 20 ads found
Page 17. 20 ads found
Page 18. 20 ads found
Page 19. 20 ads found
Page 20. 20 ads found
Page 21. 20 ads found
Page 22. 20 ads found
Page 23. 20 ads found
Page 24. 20 ads found
Page 25. 20 ads found
Page 26. 19 ads found
Page 27. 20 ads found
Page 28. 20 ads found
Page 29. 20 ads found
Page 30. 20 ads found
Page 31. 20 ads found
Page 32. 20 ads found
Page 33. 20 ads found
Page 34. 20 ads found
Page 35. 20 ads found
Page 36. 20 ads found
Page 37. 20 ads found
Page 38. 20 ads found
Page 39. 20 ads found
Page 40. 20 ads found
Page 41. 20 ads found
Page 42. 20 ads found
Page 43. 20 ads found
Page 44. 20 ads found
Page 45. 20 ads found
Page 46. 20 ads fou